# Quickstart

Model, dataset, one batch. Needs `data/openfake_train.csv` from `scripts/run_train.py`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import yaml, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.model import Detector
from src.losses import patch_loss
from src.data import TrainDataset, load_rows

cfg = yaml.safe_load(open('configs/train.yaml'))

In [ ]:
# model: frozen DINOv3 + small head on patch features
model = Detector(**cfg['model'])
print('patch', model.patch_size, '| head params', sum(p.numel() for p in model.head.parameters()))

In [ ]:
# dataset: real and generated images, one row per image
rows = load_rows('data/openfake_train.csv')
dataset = TrainDataset(rows, crop=cfg['train']['crop'] // model.patch_size * model.patch_size)
batch = next(iter(DataLoader(dataset, batch_size=4, shuffle=True)))
batch['image'].shape, batch['label']

In [ ]:
# forward and loss
patch_logits, image_logit = model(batch['image'])
loss = patch_loss(patch_logits, image_logit, batch['target'])
patch_logits.shape, image_logit, loss.item()

In [ ]:
# image, target map, predicted map
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(batch['image'][0].permute(1, 2, 0))
ax[1].imshow(batch['target'][0, 0], vmin=0, vmax=1)
ax[2].imshow(patch_logits[0, 0].sigmoid().detach(), vmin=0, vmax=1)
for a, title in zip(ax, ['image', 'target', 'prediction']):
    a.set_title(title); a.axis('off')